In [5]:
import time
import category_encoders as ce
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import numpy as np

dataset = pd.read_csv(r'C:\Users\Alicja\Desktop\kurs-datascience\DataCoSupplyChainDataset.csv', encoding='latin1')

tłumaczenie_kolumn = {
    'Type': 'Typ_platnosci', 'Days for shipping (real)': 'Rzeczywiste_dni_wysylki',
    'Days for shipment (scheduled)': 'Planowane_dni_wysylki', 'Benefit per order': 'Zysk_na_zamowienie',
    'Sales per customer': 'Sprzedaz_na_klienta', 'Delivery Status': 'Status_dostawy',
    'Late_delivery_risk': 'Ryzyko_opoznienia', 'Category Name': 'Nazwa_kategorii',
    'Customer City': 'Miasto_klienta', 'Customer Country': 'Kraj_klienta',
    'Customer State': 'Stan_klienta', 'Department Name': 'Nazwa_dzialu',
    'Market': 'Rynek', 'Order City': 'Miasto_zamowienia', 'Order Country': 'Kraj_zamowienia',
    'order date (DateOrders)': 'Data_zamowienia', 'Order Item Discount': 'Rabat_pozycji',
    'Order Item Discount Rate': 'Stopa_rabatu_pozycji', 'Order Item Product Price': 'Cena_produktu_w_zamowieniu',
    'Order Item Profit Ratio': 'Wspolczynnik_zysku_pozycji', 'Order Item Quantity': 'Ilosc_pozycji',
    'Sales': 'Sprzedaz_brutto', 'Order Item Total': 'Suma_pozycji_netto', 'Order Status': 'Status_zamowienia',
    'Product Name': 'Nazwa_produktu', 'Product Price': 'Cena_produktu',
    'Shipping date (DateOrders)': 'Data_wysylki', 'Shipping Mode': 'Tryb_wysylki'
}
dataset.rename(columns=tłumaczenie_kolumn, inplace=True)

dataset['Data_zamowienia'] = pd.to_datetime(dataset['Data_zamowienia'], format='%m/%d/%Y %H:%M')
dataset['Dzien_tygodnia_zamowienia'] = dataset['Data_zamowienia'].dt.dayofweek
dataset['Miesiac_zamowienia'] = dataset['Data_zamowienia'].dt.month

cechy_numeryczne = ['Planowane_dni_wysylki', 'Cena_produktu', 'Ilosc_pozycji', 'Rabat_pozycji',
                     'Stopa_rabatu_pozycji', 'Dzien_tygodnia_zamowienia', 'Miesiac_zamowienia']
cechy_onehot = ['Tryb_wysylki', 'Rynek', 'Typ_platnosci', 'Nazwa_dzialu']
cechy_target_encoding = ['Miasto_klienta', 'Kraj_klienta', 'Stan_klienta',
                          'Miasto_zamowienia', 'Kraj_zamowienia', 'Nazwa_kategorii', 'Nazwa_produktu']

X = dataset[cechy_numeryczne + cechy_onehot + cechy_target_encoding]
y = dataset['Ryzyko_opoznienia']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train = pd.get_dummies(X_train, columns=cechy_onehot)
X_test = pd.get_dummies(X_test, columns=cechy_onehot)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

encoder = ce.TargetEncoder(cols=cechy_target_encoding, smoothing=10)
X_train[cechy_target_encoding] = encoder.fit_transform(X_train[cechy_target_encoding], y_train)
X_test[cechy_target_encoding] = encoder.transform(X_test[cechy_target_encoding])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preprocessing gotowy:", X_train.shape, X_test.shape)

modele = {
    "Baseline (Dummy)": DummyClassifier(strategy='most_frequent', random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

wyniki = {}
for nazwa, model in modele.items():
    start = time.time()
    if nazwa == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    czas = time.time() - start
    acc = accuracy_score(y_test, y_pred)
    wyniki[nazwa] = {"accuracy": acc, "czas_s": czas, "model": model}
    print(f"{nazwa}: accuracy={acc:.4f}, czas={czas:.2f}s")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_rf = cross_val_score(wyniki["Random Forest"]["model"], X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"\nCV Random Forest: {scores_rf.mean():.4f} ± {scores_rf.std():.4f}")

mlflow.set_experiment("Predykcja_Opoznien_Dostaw")

for nazwa, dane in wyniki.items():
    with mlflow.start_run(run_name=nazwa):
        model = dane["model"]
        mlflow.log_params(model.get_params())
        mlflow.log_metric("accuracy", dane["accuracy"])
        mlflow.log_metric("czas_treningu_s", dane["czas_s"])
        if nazwa == "Random Forest":
            mlflow.log_metric("accuracy_cv_mean", scores_rf.mean())
            mlflow.log_metric("accuracy_cv_std", scores_rf.std())
        mlflow.sklearn.log_model(model, "model")
        print(f"Zalogowano: {nazwa}")

print("\nWszystkie eksperymenty zalogowane w MLflow!")

Preprocessing gotowy: (144415, 38) (36104, 38)
Baseline (Dummy): accuracy=0.5483, czas=0.02s
Logistic Regression: accuracy=0.7027, czas=0.69s
Random Forest: accuracy=0.7799, czas=37.66s
Gradient Boosting: accuracy=0.7049, czas=107.68s

CV Random Forest: 0.7746 ± 0.0021


2026/08/01 18:05:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/01 18:06:33 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Zalogowano: Baseline (Dummy)


2026/08/01 18:06:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/01 18:07:07 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Zalogowano: Logistic Regression


2026/08/01 18:07:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/01 18:08:04 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/08/01 18:08:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Zalogowano: Random Forest


2026/08/01 18:08:40 WARNING mlflow.utils.requirements_utils: Found torch version (2.13.0+cpu) contains a local version label (+cpu). MLflow logged a pip requirement for this package as 'torch==2.13.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Zalogowano: Gradient Boosting

Wszystkie eksperymenty zalogowane w MLflow!
